# 03 - Mapeamento de Criticas

Este notebook extrai as regras de crítica dos arquivos de metadados, expande condições derivadas e aplica as regras sobre os dados já existentes no banco configurado em INPUT_DB.

Resultado final deste fluxo:
- tabela criticas atualizada no banco INPUT_DB
- tabela estabelecimentos_criticas com as ocorrências encontradas

In [1]:
import json
import re
import sqlite3
from pathlib import Path
from glob import glob

import pandas as pd

## 1. Imports

In [2]:
# ==============================
# Configuração (preencha antes de executar)
# ==============================

INPUT_GLOB = None
INPUT_DB = None
VARIABLE_MAP_PATH = None

# Controle de escrita no banco INPUT_DB
OVERWRITE_TABLES = True

# Colunas de chave para vincular estabelecimentos às críticas
KEY_COLS = []

# Lista opcional de variáveis tratadas como string durante o eval
STRING_VARS = []

# Nome da tabela com os dados de estabelecimentos para mapeamento das críticas
ESTABEL_TABLE = ""

# Mapeamento da fonte de dados para a variável de código do produto
PRODUCT_CODE_BY_SOURCE = {}

# Configuração privada opcional (não versionada)
PRIVATE_CONFIG_PATH = "../config/mapeamento_criticas.private.json"

# --- não edite abaixo desta linha ---
_PRIVATE = Path(PRIVATE_CONFIG_PATH)
if _PRIVATE.exists():
    _cfg = json.loads(_PRIVATE.read_text(encoding="utf-8"))
    INPUT_GLOB = INPUT_GLOB or _cfg.get("input_glob")
    INPUT_DB = INPUT_DB or _cfg.get("input_db")
    VARIABLE_MAP_PATH = VARIABLE_MAP_PATH or _cfg.get("variable_map_path")
    OVERWRITE_TABLES = _cfg.get("overwrite_tables", OVERWRITE_TABLES)
    KEY_COLS = _cfg.get("key_cols", KEY_COLS)
    ESTABEL_TABLE = _cfg.get("estabel_table", ESTABEL_TABLE)
    STRING_VARS = _cfg.get("string_vars", STRING_VARS)
    PRODUCT_CODE_BY_SOURCE = _cfg.get("product_code_by_source", PRODUCT_CODE_BY_SOURCE)
    print("Configuração privada carregada.")

assert INPUT_GLOB is not None, "Configure INPUT_GLOB antes de executar."
assert INPUT_DB is not None, "Configure INPUT_DB antes de executar."
assert VARIABLE_MAP_PATH is not None, "Configure VARIABLE_MAP_PATH antes de executar."

_INPUT_DB = Path(INPUT_DB)
_VARIABLE_MAP_PATH = Path(VARIABLE_MAP_PATH)
_QUADROS = sorted(glob(INPUT_GLOB))

assert _INPUT_DB.exists(), f"Banco de entrada não encontrado: {_INPUT_DB}"
assert _VARIABLE_MAP_PATH.exists(), f"Mapa de variáveis não encontrado: {_VARIABLE_MAP_PATH}"
assert _QUADROS, f"Nenhum arquivo encontrado com INPUT_GLOB={INPUT_GLOB}"

print(f"Banco alvo (INPUT_DB): {_INPUT_DB.resolve()}")
print(f"Mapa de variáveis: {_VARIABLE_MAP_PATH.resolve()}")
print(f"Arquivos de quadros: {len(_QUADROS)}")
print(f"Sobrescrever tabelas: {OVERWRITE_TABLES}")
print(f"Mapeamentos tabela->código de produto: {len(PRODUCT_CODE_BY_SOURCE)}")

Configuração privada carregada.
Banco alvo (INPUT_DB): /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/amstr_geral.db
Mapa de variáveis: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data_example/mapa_variaveis.json
Arquivos de quadros: 3
Sobrescrever tabelas: True
Mapeamentos tabela->código de produto: 2


## 2. Configuracao e Validacoes

Use variaveis na celula de configuracao ou um arquivo privado em config/mapeamento_criticas.private.json.

In [3]:
def collect_criticas_from_quadros(quadros_files: list[str]) -> pd.DataFrame:
    """Lê arquivos de quadros e monta o DataFrame base de críticas."""
    all_crit = []

    for q_path in quadros_files:
        with open(q_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        cond_quadro = data.get("CondicaoDeExibicao", "")

        blocos = []
        blocos.extend(data.get("Quesitos", []))
        blocos.extend(data.get("QuesitoProdutos", []))
        blocos.extend(data.get("ListaProdutos", []))
        blocos.extend(data.get("QuesitoProdutosAgrupados", []))

        for bloco in blocos:
            cond_bloco = bloco.get("CondicaoDeExibicao", "")
            for pergunta in bloco.get("Perguntas", []):
                cond_pergunta = pergunta.get("CondicaoDeExibicao", "")
                for critica in pergunta.get("Criticas", []):
                    if "Condicao" not in critica:
                        continue

                    # Concatena as condições de exibição em ordem de escopo.
                    partes = []
                    if cond_quadro:
                        partes.append(f"({cond_quadro})")
                    if cond_bloco:
                        partes.append(f"({cond_bloco})")
                    if cond_pergunta:
                        partes.append(f"({cond_pergunta})")
                    partes.append(f"({critica['Condicao']})")

                    all_crit.append({
                        "quadro": Path(q_path).stem.replace("quadro", ""),
                        "pergunta": pergunta.get("Titulo", ""),
                        "variavel": pergunta.get("CodigoVariavel", ""),
                        "id_origem": critica.get("Id", ""),
                        "condicao": critica.get("Condicao", ""),
                        "condicao_completa": " && ".join(partes),
                        "tipo": critica.get("Tipo", ""),
                        "mensagem": critica.get("Mensagem", ""),
                    })

    df = pd.DataFrame(all_crit).drop_duplicates().reset_index(drop=True)
    if not df.empty:
        df["quadro"] = pd.to_numeric(df["quadro"], errors="coerce").fillna(0).astype(int)
    return df


mapa_variaveis = json.loads(_VARIABLE_MAP_PATH.read_text(encoding="utf-8"))
df_criticas = collect_criticas_from_quadros(_QUADROS)

print(f"Críticas coletadas: {len(df_criticas)}")
df_criticas.head()

Críticas coletadas: 9


,quadro,pergunta,variavel,id_origem,condicao,condicao_completa,tipo,mensagem
0,1,Telefone Fixo,V01150101,personalizada,V01150000 == 2 && V01150101 == null && V011501...,(V01150000 == 2) && (V01150000 == 2 && V011501...,erro,Registre pelo menos um numero
1,1,Area,V01170100,personalizada,V01170100 == 0 && V01170500 != 2,(V01170100 == 0 && V01170500 != 2),erro,Deve ser preenchida a area total
2,1,Area,V01170100,personalizada,VW01170300 > 10000,(VW01170300 > 10000),alerta,Area total maior que 10.000 hectares.
3,34,Pes existentes,V39020300,personalizada,(X_V39020300 / VW04100000) > getDensidadeMaxim...,(V33010600 == 2) && ((X_V39020300 / VW04100000...,alerta,Densidade fora dos limites.
4,34,Pes existentes,V39020300,personalizada,X_V39020300 == 0,(V33010600 == 2) && (X_V39020300 == 0),erro,Preenchimento obrigatorio.


## 3. Extracao das Criticas dos Quadros

Esta etapa monta o dataframe base com todas as criticas e as condicoes completas por escopo.

In [4]:
def clean_condition(condition: str) -> str:
    """Remove prefixo X_ de variáveis no formato X_V... e X_VW..."""
    return re.sub(r"X_(VW?\d+)", r"\1", str(condition))


def expand_recursive(condition: str, var_map: dict, max_depth: int = 20) -> str:
    """Expande variáveis derivadas VW quando a definição está no formato VW...=...;"""
    current = str(condition)

    for _ in range(max_depth):
        vw_vars = set(re.findall(r"\b(VW\d+)\b", current))
        if not vw_vars:
            break

        replaced = False
        for var in vw_vars:
            definition = str(var_map.get(var, {}).get("condition", "")).strip()
            prefix = f"{var}="
            if definition.startswith(prefix) and definition.endswith(";"):
                rhs = definition[len(prefix):-1]
                current = re.sub(r"\b" + re.escape(var) + r"\b", f"({rhs})", current)
                replaced = True

        if not replaced:
            break

    return current


def formatar_subexpressao(tokens: list[str]) -> str:
    """Agrupa comparações aritméticas com parênteses para avaliação vetorizada."""
    if not tokens:
        return ""

    operadores_comp = {"<", ">", "<=", ">=", "==", "!="}
    operadores_arit = {"+", "-", "*", "/", "%", "**"}

    idx_comp = None
    op_comp = None
    for i, token in enumerate(tokens):
        if token in operadores_comp:
            idx_comp = i
            op_comp = token
            break

    if idx_comp is None:
        return f"( {' '.join(tokens)} )"

    esquerda = tokens[:idx_comp]
    direita = tokens[idx_comp + 1:]

    expr_esquerda = '( ' + ' '.join(esquerda) + ' )' if any(t in operadores_arit for t in esquerda) else ' '.join(esquerda)
    expr_direita = '( ' + ' '.join(direita) + ' )' if any(t in operadores_arit for t in direita) else ' '.join(direita)

    return f"( {expr_esquerda} {op_comp} {expr_direita} )"


def converter_expressao_vetorizada(tokens: list[str]) -> str:
    """Converte tokens em expressão pronta para eval vetorizado com pandas."""
    if not tokens:
        return ""

    operadores_logicos = {"&", "|"}
    resultado = []
    i = 0

    while i < len(tokens):
        token = tokens[i]
        if token in operadores_logicos:
            resultado.append(f" {token} ")
            i += 1
            continue

        sub = []
        while i < len(tokens) and tokens[i] not in operadores_logicos:
            sub.append(tokens[i])
            i += 1
        resultado.append(formatar_subexpressao(sub))

    return ''.join(resultado)


def converter_expressao_logica(expr: str, req_functions: list[str], none_fill: int = 0) -> str:
    """Normaliza operadores e tokeniza expressão para execução no Python/pandas."""
    expr = str(expr).replace("&&", "&").replace("||", "|")
    expr = expr.replace("null", str(none_fill))

    logic_op = '&|\\|'
    pattern = r'''
        \bV\w+\b
      | \bNone\b
      | \b(?:{})\b
      | {}
      | [()]+
      | !=|==|>=|<=|>|<
      | [+,\-*/%]
      | \*\*
      | ,
      | \d+(?:\.\d+)?
    '''.format('|'.join(req_functions), logic_op)

    tokens = re.findall(pattern, expr, flags=re.VERBOSE)
    return converter_expressao_vetorizada(tokens)


# Cria colunas de condição limpa e condição final expandida
df_criticas["condicao_limpa"] = df_criticas["condicao_completa"].apply(clean_condition)
df_criticas["condicao_final"] = df_criticas["condicao_limpa"].apply(lambda x: expand_recursive(x, mapa_variaveis))

# Adicionalmente, processa a condição base original para comparação
df_criticas["condicao_base_limpa"] = df_criticas["condicao"].apply(clean_condition)
df_criticas["condicao_base_final"] = df_criticas["condicao_base_limpa"].apply(lambda x: expand_recursive(x, mapa_variaveis))

## 4. Normalizacao e Expansao das Condicoes

Remove prefixos tecnicos e expande variaveis derivadas VW quando possivel.

In [5]:
# Funções auxiliares utilizadas pelas expressões das críticas
NONE_FILL = 0

def build_product_limits(variable_map: dict, source_to_product_var: dict[str, str]) -> dict[int, dict[str, float]]:
    """Monta lookup de limiares por código de produto a partir do mapa de variáveis."""
    prod_limits: dict[int, dict[str, float]] = {}

    for prod_var in sorted(set(source_to_product_var.values())):
        for item in variable_map.get(prod_var, {}).get("possible_values", []):
            cod = str(item.get("cod", "")).strip()
            if not cod.isdigit():
                continue

            parsed: dict[str, float] = {}
            thresholds = item.get("thresholds", {})
            for key in ["min", "max", "minDensity", "maxDensity", "minYield", "maxYield"]:
                raw = thresholds.get(key, "")
                parsed[key] = float(raw) if str(raw).strip() else 0.0

            prod_limits[int(cod)] = parsed

    return prod_limits


PROD_LIMITS = build_product_limits(mapa_variaveis, PRODUCT_CODE_BY_SOURCE)
limit_max_map = {k: v.get("max", 0.0) for k, v in PROD_LIMITS.items()}
limit_min_map = {k: v.get("min", 0.0) for k, v in PROD_LIMITS.items()}
limit_max_density_map = {k: v.get("maxDensity", 0.0) for k, v in PROD_LIMITS.items()}
limit_min_density_map = {k: v.get("minDensity", 0.0) for k, v in PROD_LIMITS.items()}
limit_max_yield_map = {k: v.get("maxYield", 0.0) for k, v in PROD_LIMITS.items()}
limit_min_yield_map = {k: v.get("minYield", 0.0) for k, v in PROD_LIMITS.items()}

def getPrecoMaximo(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_max_map).fillna(0)
    return limit_max_map.get(cod_prod, 0)

def getPrecoMinimo(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_min_map).fillna(0)
    return limit_min_map.get(cod_prod, 0)

def getDensidadeMaxima(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_max_density_map).fillna(0)
    return limit_max_density_map.get(cod_prod, 0)

def getDensidadeMinima(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_min_density_map).fillna(0)
    return limit_min_density_map.get(cod_prod, 0)

def getRendimentoMaximo(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_max_yield_map).fillna(0)
    return limit_max_yield_map.get(cod_prod, 0)

def getRendimentoMinimo(cod_prod):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.map(limit_min_yield_map).fillna(0)
    return limit_min_yield_map.get(cod_prod, 0)

def getUf(id_setor):
    if isinstance(id_setor, pd.Series):
        return id_setor.astype(str).str[:2]
    return str(id_setor)[:2]

def codigoProdutoNoIntervalo(cod_prod, min_value, max_value):
    if isinstance(cod_prod, pd.Series):
        return cod_prod.between(min_value, max_value)
    return min_value <= cod_prod <= max_value

def numeroProdutosExistentes(cod_prods, start_cod_prod: int, end_cod_prod: int):
    alvo = set(str(x) for x in range(start_cod_prod, end_cod_prod + 1))

    def _count(val):
        if pd.isna(val) or val in (0, "0", ""):
            return 0
        if isinstance(val, str):
            itens = set(x for x in val.split(",") if x)
        else:
            itens = {str(val)}
        return len(itens.intersection(alvo))

    if isinstance(cod_prods, pd.Series):
        return cod_prods.apply(_count)
    return _count(cod_prods)

print(f"Limiares carregados para {len(PROD_LIMITS)} códigos de produto.")

Limiares carregados para 0 códigos de produto.


## 5. Funcoes Auxiliares do Motor

Define funcoes de limite por produto e utilitarios usados nas expressoes das criticas.

In [6]:
def get_database_schema(conn: sqlite3.Connection, key_cols: list[str]) -> dict[str, list[str]]:
    """Lê colunas de cada tabela, removendo as chaves de vinculação."""
    schema: dict[str, list[str]] = {}
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()

    for (table,) in tables:
        cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
        schema[table] = [c[1] for c in cols if c[1] not in key_cols]

    return schema


def map_structured_conditions(df: pd.DataFrame, db_schema: dict[str, list[str]]) -> list[dict]:
    """Mapeia cada condição para as tabelas necessárias com base nas variáveis V..."""
    structured = []
    pattern = r"\bV\d+\b"

    for idx, row in df.iterrows():
        cond = str(row["condicao_final"])
        vars_found = set(re.findall(pattern, cond))
        data_source: dict[str, list[str]] = {}

        for var in vars_found:
            for table, cols in db_schema.items():
                if var in cols:
                    data_source.setdefault(table, []).append(var)
                    break

        structured.append({
            "id_critica": int(idx),
            "quadro": int(row["quadro"]),
            "tipo": str(row["tipo"]),
            "mensagem": str(row["mensagem"]),
            "condicao": str(row["condicao"]),
            "condicao_limpa": str(row["condicao_limpa"]),
            "condicao_final": cond,
            "data_source": data_source,
        })

    return structured


def get_data_pandas(conn: sqlite3.Connection, data_source: dict[str, list[str]], key_cols: list[str]) -> pd.DataFrame:
    """Monta SELECT + JOIN entre tabelas necessárias e retorna DataFrame preenchido com 0."""
    tables = list(data_source.keys())
    if not tables:
        return pd.DataFrame()

    base = tables[0]
    select_fields = [f"{base}.{k}" for k in key_cols]

    for table, cols in data_source.items():
        for col in sorted(set(cols)):
            if col not in key_cols:
                select_fields.append(f"{table}.{col}")

    query = f"SELECT {', '.join(select_fields)} FROM {base}"
    for table in tables[1:]:
        join_on = " AND ".join([f"{base}.{k} = {table}.{k}" for k in key_cols])
        query += f" INNER JOIN {table} ON {join_on}"

    try:
        out = pd.read_sql(query, conn)
        return out.fillna(NONE_FILL)
    except Exception:
        return pd.DataFrame()


def corrigir_codigo_variavel_para_contagem_produtos(condicao: str) -> tuple[str, set[str]]:
    """Ajusta numeroProdutosExistentes(V...) para numeroProdutosExistentes(V..._L)."""
    cond = str(condicao)
    alteradas = set()
    padrao = re.compile(r"numeroProdutosExistentes\((V\d{8})")

    for m in padrao.finditer(cond):
        var_cod = m.group(1)
        if var_cod in alteradas:
            continue
        cond = cond.replace(
            f"numeroProdutosExistentes({var_cod}",
            f"numeroProdutosExistentes({var_cod}_L"
        )
        alteradas.add(var_cod)

    return cond, alteradas


def append_count_columns(
    conn: sqlite3.Connection,
    data_source: dict[str, list[str]],
    key_cols: list[str],
    df_source: pd.DataFrame,
    variaveis_contagem: set[str],
) -> pd.DataFrame:
    """Cria colunas *_L (lista de códigos) para suporte a numeroProdutosExistentes."""
    for v in variaveis_contagem:
        table_found = None
        for table, vars_table in data_source.items():
            if v in vars_table:
                table_found = table
                break

        if table_found is None:
            continue

        query = f"""
SELECT
  {','.join(key_cols)},
  GROUP_CONCAT({v}) as {v}_L
FROM {table_found}
GROUP BY {','.join(key_cols)}
        """
        try:
            df_cnt = pd.read_sql(query, conn).fillna(NONE_FILL)
            df_source = pd.merge(df_source, df_cnt, on=key_cols, how="left")
        except Exception:
            continue

    return df_source

print("Funções de acesso ao banco e mapeamento carregadas.")

Funções de acesso ao banco e mapeamento carregadas.


## 6. Mapeamento das Condicoes para Tabelas do Banco

Relaciona variaveis de cada condicao com as tabelas reais do banco SQLite.

In [7]:
REQ_FUNCTIONS = [
    "getPrecoMaximo",
    "getPrecoMinimo",
    "getDensidadeMaxima",
    "getDensidadeMinima",
    "getRendimentoMaximo",
    "getRendimentoMinimo",
    "getUf",
    "codigoProdutoNoIntervalo",
    "numeroProdutosExistentes",
]
NOT_IMPLEMENTED_FUNCTIONS = ["condicaoVerdadeiraParaAlgumProduto"]

def resolve_product_var_for_source(source: str, source_to_product_var: dict[str, str]) -> str | None:
    """Resolve o código de produto para a fonte, com fallback entre nomes com/sem prefixo amstr_."""
    source_norm = str(source).strip()
    source_lower = source_norm.lower()

    candidates = [source_norm, source_lower]
    if source_lower.startswith("amstr_"):
        candidates.append(source_lower.removeprefix("amstr_"))
    else:
        candidates.append(f"amstr_{source_lower}")

    for key in candidates:
        if key in source_to_product_var:
            return source_to_product_var[key]

    return None


def ensure_product_code_in_data_source(
    data_source: dict[str, list[str]],
    source_to_product_var: dict[str, str],
) -> tuple[dict[str, list[str]], str | None, str | None]:
    """Adiciona ao data_source a variável de código do produto da fonte utilizada."""
    out: dict[str, list[str]] = {
        table: list(dict.fromkeys(cols)) for table, cols in data_source.items()
    }
    reference_variable = None

    for source in out:
        ref_var = resolve_product_var_for_source(source, source_to_product_var)
        if not ref_var:
            continue

        if ref_var not in out[source]:
            out[source].append(ref_var)

        if reference_variable is None:
            reference_variable = ref_var

    return out, reference_variable, None


def collect_candidate_product_vars(data_source: dict[str, list[str]], source_to_product_var: dict[str, str], reference_variable: str | None) -> list[str]:
    """Lista variáveis candidatas de código de produto para as fontes usadas na condição."""
    candidates: list[str] = []
    if reference_variable:
        candidates.append(reference_variable)

    for source in data_source:
        ref_var = resolve_product_var_for_source(source, source_to_product_var)
        if ref_var and ref_var not in candidates:
            candidates.append(ref_var)

    return candidates


def get_product_code_series(df_rows: pd.DataFrame, reference_variable: str | None):
    """Busca a coluna de código de produto considerando pequenas variações de nome."""
    if not reference_variable:
        return None

    if reference_variable in df_rows.columns:
        return df_rows[reference_variable]

    ref_lower = reference_variable.lower()
    for col in df_rows.columns:
        col_lower = str(col).lower()
        if (
            col_lower == ref_lower
            or col_lower.endswith(f".{ref_lower}")
            or col_lower.endswith(f"_{ref_lower}")
        ):
            return df_rows[col]

    return None


def evaluate_single_condition(
    conn: sqlite3.Connection,
    cond_data: dict,
    key_cols: list[str],
    string_vars: list[str],
) -> tuple[list[tuple], str | None]:
    """Avalia uma condição e retorna ocorrências no formato das chaves + id_critica + cod_produto."""
    cond = str(cond_data["condicao_final"])
    if any(fn in cond for fn in NOT_IMPLEMENTED_FUNCTIONS):
        return [], "função não implementada"

    cond_adj, vars_count = corrigir_codigo_variavel_para_contagem_produtos(cond)
    data_source, reference_variable, source_status = ensure_product_code_in_data_source(
        cond_data["data_source"], PRODUCT_CODE_BY_SOURCE
    )

    if source_status:
        return [], source_status

    if not data_source:
        return [], "sem data_source"

    df = get_data_pandas(conn, data_source, key_cols)
    if df.empty:
        return [], "sem dados"

    if vars_count:
        df = append_count_columns(conn, data_source, key_cols, df, vars_count)

    cols_to_skip = set(string_vars)
    for col in df.columns:
        if col in cols_to_skip or col.endswith("_L"):
            continue

        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue

    local_env = {col: df[col] for col in df.columns}
    local_env.update({
        "getPrecoMaximo": getPrecoMaximo,
        "getPrecoMinimo": getPrecoMinimo,
        "getDensidadeMaxima": getDensidadeMaxima,
        "getDensidadeMinima": getDensidadeMinima,
        "getRendimentoMaximo": getRendimentoMaximo,
        "getRendimentoMinimo": getRendimentoMinimo,
        "codigoProdutoNoIntervalo": codigoProdutoNoIntervalo,
        "numeroProdutosExistentes": numeroProdutosExistentes,
        "getUf": getUf,
        "None": NONE_FILL,
    })

    expr = converter_expressao_logica(cond_adj, REQ_FUNCTIONS, none_fill=NONE_FILL)
    expr = re.sub(r"(?<!\.)\b0+(\d+)", r"\1", expr)

    try:
        mask = eval(expr, {}, local_env)
        offending_rows = df[mask]
    except Exception as ex:
        return [], f"erro eval: {type(ex).__name__}"

    if offending_rows.empty:
        return [], None

    out = offending_rows[key_cols].copy()
    out["id_critica"] = int(cond_data["id_critica"] )
    product_code_series = None
    for candidate_var in collect_candidate_product_vars(data_source, PRODUCT_CODE_BY_SOURCE, reference_variable):
        candidate_series = get_product_code_series(offending_rows, candidate_var)
        if candidate_series is None:
            continue
        if product_code_series is None:
            product_code_series = candidate_series
        else:
            product_code_series = product_code_series.combine_first(candidate_series)

    out["cod_produto"] = product_code_series if product_code_series is not None else None

    return list(out.itertuples(index=False, name=None)), None


with sqlite3.connect(_INPUT_DB) as conn:
    db_schema = get_database_schema(conn, KEY_COLS)

structured_cond = map_structured_conditions(df_criticas, db_schema)
print(f"Condições estruturadas: {len(structured_cond)}")

occurrences: list[tuple] = []
logs_execucao = []

with sqlite3.connect(_INPUT_DB) as conn:
    for cond in structured_cond:
        occ, status = evaluate_single_condition(conn, cond, KEY_COLS, STRING_VARS)
        occurrences.extend(occ)
        logs_execucao.append({
            "id_critica": cond["id_critica"],
            "status": status or "ok",
            "ocorrencias": len(occ),
        })

df_logs_execucao = pd.DataFrame(logs_execucao)
print(f"Ocorrências encontradas: {len(occurrences)}")
df_logs_execucao["status"].value_counts().to_frame("qtd")

Condições estruturadas: 9
Ocorrências encontradas: 4


,qtd
status,
ok,6
erro eval: NameError,3


## 7. Execucao do Motor de Criticas

Avalia as condicoes sobre os dados do banco INPUT_DB e coleta as ocorrencias por estabelecimento.

In [8]:
# Persistência das tabelas no banco INPUT_DB
criticas_to_save = df_criticas[[
    "quadro",
    "tipo",
    "mensagem",
    "condicao",
    "condicao_limpa",
    "condicao_final",
]].copy()
criticas_to_save = criticas_to_save.reset_index().rename(columns={"index": "id"})

rel_cols = KEY_COLS + ["id_critica", "cod_produto"]
df_rel = pd.DataFrame(occurrences, columns=rel_cols).drop_duplicates().reset_index(drop=True)

with sqlite3.connect(_INPUT_DB) as conn:
    cur = conn.cursor()

    if OVERWRITE_TABLES:
        cur.execute("DROP TABLE IF EXISTS estabel_criticas")
        cur.execute("DROP TABLE IF EXISTS criticas")

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS criticas (
            id INTEGER PRIMARY KEY,
            quadro INTEGER,
            tipo TEXT,
            mensagem TEXT,
            condicao TEXT,
            condicao_limpa TEXT,
            condicao_final TEXT
        )
        """
    )

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS estabel_criticas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            V010100 INTEGER,
            NUM_QUADRA INTEGER,
            NUM_FACE INTEGER,
            V010800 INTEGER,
            id_critica INTEGER,
            cod_produto INTEGER,
            FOREIGN KEY (id_critica) REFERENCES criticas(id)
        )
        """
    )

    if OVERWRITE_TABLES:
        cur.execute("DELETE FROM criticas")
        cur.execute("DELETE FROM estabel_criticas")

    criticas_to_save.to_sql("criticas", conn, if_exists="append", index=False)
    if not df_rel.empty:
        df_rel.to_sql("estabel_criticas", conn, if_exists="append", index=False)

    qtd_criticas = conn.execute("SELECT COUNT(*) FROM criticas").fetchone()[0]
    qtd_rel = conn.execute("SELECT COUNT(*) FROM estabel_criticas").fetchone()[0]

print(f"Tabela criticas: {qtd_criticas} registros")
print(f"Tabela estabel_criticas: {qtd_rel} registros")
print("Processo concluído no banco configurado em INPUT_DB.")

Tabela criticas: 9 registros
Tabela estabel_criticas: 4 registros
Processo concluído no banco configurado em INPUT_DB.
